# LeetCode #73 — Set Matrix Zeroes: đánh dấu tại chỗ với bộ nhớ O(1)

**Mức độ:** Medium · **Chủ đề:** Array, Hash Table, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def setZeroes(self, matrix: List[List[int]]) -> None:
        """
        Do not return anything, modify matrix in-place instead.
        """
```

Mục tiêu của notebook:

1. Hiểu vì sao **vừa duyệt vừa gán 0** gây "lan truyền" sai.
2. Tách thành hai pha: **đánh dấu** rồi **ghi**.
3. Đi từ bộ nhớ `O(m·n)` → `O(m + n)` → `O(1)` bằng cách dùng **hàng 0 và cột 0 làm sổ đánh dấu**.
4. Xử lý chi tiết khó nhất: ô `matrix[0][0]` bị dùng chung.

# 1. Bài toán là gì?

Cho ma trận `m × n`. Nếu một phần tử bằng `0`, đặt **cả hàng và cả cột** của nó thành `0`. Làm **tại chỗ (in-place)**.

**Constraints (theo đề):**

- `m == matrix.length`, `n == matrix[0].length`
- `1 <= m, n <= 200`
- `-2^31 <= matrix[i][j] <= 2^31 - 1`

**Follow up (theo đề):** lời giải `O(m·n)` bộ nhớ là tệ; `O(m + n)` tốt hơn nhưng chưa tối ưu; bạn có làm được với **bộ nhớ hằng** không?

---

## Ví dụ

```text
1 1 1        1 0 1
1 0 1   ->   0 0 0
1 1 1        1 0 1

0 1 2 0      0 0 0 0
3 4 5 2  ->  0 4 5 0
1 3 1 5      0 3 1 0
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ma trận (mảng hai chiều) trong Python
2. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [1]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [2]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [3]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [4]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.2 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [5]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [6]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 316.53 ms | trên set: 0.075 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [7]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [8]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.3 Áp dụng vào bài này

Cách dễ: duyệt **ma trận**, ghi các hàng và cột có số 0 vào hai **set**, rồi lượt thứ hai đặt 0. Cách `O(1)` bộ nhớ: dùng chính **hàng 0 và cột 0** làm "sổ ghi", cẩn thận xử lý riêng hàng/cột đầu.

# 3. Cách ngây thơ (sai): gặp 0 là xóa ngay

In [9]:
from typing import List
import copy

def wrong_immediate(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    for r in range(m):
        for c in range(n):
            if matrix[r][c] == 0:               # SAI: số 0 vừa gán cũng bị coi là 0 "gốc"
                for k in range(n):
                    matrix[r][k] = 0
                for k in range(m):
                    matrix[k][c] = 0

def show(mat):
    for row in mat:
        print("  " + " ".join(f"{x:>2}" for x in row))

A = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
wrong_immediate(A)
show(A)

   1  0  0
   0  0  0
   0  0  0


Đáp án đúng chỉ có 5 số 0 (hàng 1 và cột 1), nhưng ở đây có 8! Số `0` mới gán ở `(1, 2)` được duyệt sau và bị coi là số 0 gốc, nên xóa luôn cột 2; tương tự `(2, 1)` xóa luôn hàng 2 — số 0 cứ thế lan ra.

> Phải **phân biệt** số 0 gốc với số 0 mới gán → tách thành hai pha: **ghi nhớ** hàng/cột cần xóa, **rồi** mới xóa.

# 4. Cách 1: bản sao — `O(m·n)` bộ nhớ

Đọc từ bản sao (chưa bị sửa), ghi vào ma trận gốc.

In [10]:
def set_zeroes_copy(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    original = [row[:] for row in matrix]
    for r in range(m):
        for c in range(n):
            if original[r][c] == 0:
                for k in range(n):
                    matrix[r][k] = 0
                for k in range(m):
                    matrix[k][c] = 0

A = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
set_zeroes_copy(A)
show(A)

   1  0  1
   0  0  0
   1  0  1


# 5. Cách 2: hai tập hàng / cột — `O(m + n)` bộ nhớ

Ta chỉ cần biết **hàng nào** và **cột nào** có số 0 — không cần cả bản sao.

In [11]:
def set_zeroes_sets(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    zero_rows, zero_cols = set(), set()
    for r in range(m):
        for c in range(n):
            if matrix[r][c] == 0:
                zero_rows.add(r)
                zero_cols.add(c)
    for r in range(m):
        for c in range(n):
            if r in zero_rows or c in zero_cols:
                matrix[r][c] = 0

A = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
set_zeroes_sets(A)
show(A)

   0  0  0  0
   0  4  5  0
   0  3  1  0


# 6. Ý tưởng chính: `O(1)` — dùng hàng 0 và cột 0 làm "sổ đánh dấu"

Thay vì hai mảng `zero_rows` (độ dài `m`) và `zero_cols` (độ dài `n`), dùng **chính** cột 0 và hàng 0 của ma trận:

- `matrix[r][0] == 0` nghĩa là "hàng `r` cần xóa".
- `matrix[0][c] == 0` nghĩa là "cột `c` cần xóa".

**Vấn đề:** hàng 0 và cột 0 vừa là dữ liệu, vừa là sổ. Ô `matrix[0][0]` thuộc **cả hai** — nó không thể vừa nói "hàng 0 cần xóa" vừa nói "cột 0 cần xóa". Giải pháp: dùng **một biến riêng** `first_col_zero` cho cột 0; ô `matrix[0][0]` chỉ dành cho hàng 0.

Các bước:

1. Quét toàn bộ: gặp `matrix[r][c] == 0` thì đánh dấu `matrix[r][0] = 0` (nếu `c > 0`) và `matrix[0][c] = 0`; nếu `c == 0` thì `first_col_zero = True`.
2. Xóa phần **bên trong** (`r ≥ 1`, `c ≥ 1`) theo sổ.
3. Xóa hàng 0 nếu `matrix[0][0] == 0`.
4. Xóa cột 0 nếu `first_col_zero`.

Thứ tự 2 → 3 → 4 quan trọng: sổ phải được **đọc xong** trước khi bị ghi đè.

In [12]:
def set_zeroes_o1(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    first_col_zero = False
    for r in range(m):
        if matrix[r][0] == 0:
            first_col_zero = True
        for c in range(1, n):
            if matrix[r][c] == 0:
                matrix[r][0] = 0
                matrix[0][c] = 0
    for r in range(1, m):
        for c in range(1, n):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    if matrix[0][0] == 0:
        for c in range(n):
            matrix[0][c] = 0
    if first_col_zero:
        for r in range(m):
            matrix[r][0] = 0

A = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
set_zeroes_o1(A)
show(A)

   0  0  0  0
   0  4  5  0
   0  3  1  0


# 7. Mô phỏng

In [13]:
def trace_set_zeroes(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    print("Ban đầu:"); show(matrix)
    first_col_zero = False
    for r in range(m):
        if matrix[r][0] == 0:
            first_col_zero = True
        for c in range(1, n):
            if matrix[r][c] == 0:
                matrix[r][0] = 0
                matrix[0][c] = 0
    print(f"Sau pha đánh dấu (first_col_zero = {first_col_zero}):"); show(matrix)
    for r in range(1, m):
        for c in range(1, n):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    print("Sau khi xóa phần bên trong:"); show(matrix)
    if matrix[0][0] == 0:
        for c in range(n):
            matrix[0][c] = 0
    if first_col_zero:
        for r in range(m):
            matrix[r][0] = 0
    print("Sau khi xử lý hàng 0 và cột 0:"); show(matrix)

trace_set_zeroes([[1, 2, 3, 4], [5, 0, 7, 8], [9, 10, 11, 0], [13, 14, 15, 16]])

Ban đầu:
   1  2  3  4
   5  0  7  8
   9 10 11  0
  13 14 15 16
Sau pha đánh dấu (first_col_zero = False):
   1  0  3  0
   0  0  7  8
   0 10 11  0
  13 14 15 16
Sau khi xóa phần bên trong:
   1  0  3  0
   0  0  0  0
   0  0  0  0
  13  0 15  0
Sau khi xử lý hàng 0 và cột 0:
   1  0  3  0
   0  0  0  0
   0  0  0  0
  13  0 15  0


| Pha | Điều xảy ra |
|---|---|
| Đánh dấu | `(1,1)=0` → `matrix[1][0]=0`, `matrix[0][1]=0`; `(2,3)=0` → `matrix[2][0]=0`, `matrix[0][3]=0` |
| Bên trong | hàng 1, 2 và cột 1, 3 bị xóa (chỉ ở phần `r,c ≥ 1`) |
| Hàng 0 | `matrix[0][0] = 1` ≠ 0 → giữ hàng 0 (đã có dấu ở cột 1, 3) |
| Cột 0 | `first_col_zero = False` → giữ cột 0 (đã có dấu ở hàng 1, 2) |

# 8. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def setZeroes(self, matrix: List[List[int]]) -> None:
        m, n = len(matrix), len(matrix[0])
        first_col_zero = False
        for r in range(m):
            if matrix[r][0] == 0:
                first_col_zero = True
            for c in range(1, n):
                if matrix[r][c] == 0:
                    matrix[r][0] = 0
                    matrix[0][c] = 0
        for r in range(1, m):
            for c in range(1, n):
                if matrix[r][0] == 0 or matrix[0][c] == 0:
                    matrix[r][c] = 0
        if matrix[0][0] == 0:
            for c in range(n):
                matrix[0][c] = 0
        if first_col_zero:
            for r in range(m):
                matrix[r][0] = 0

In [15]:
import random

def run(f, mat):
    mat = copy.deepcopy(mat)
    ret = f(mat)
    assert ret is None
    return mat

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", run(sol.setZeroes, [[1, 1, 1], [1, 0, 1], [1, 1, 1]]), [[1, 0, 1], [0, 0, 0], [1, 0, 1]])
check("Ví dụ 2", run(sol.setZeroes, [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]), [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]])
check("1 x 1 là 0", run(sol.setZeroes, [[0]]), [[0]])
check("Không có 0", run(sol.setZeroes, [[1, 2], [3, 4]]), [[1, 2], [3, 4]])
check("0 chỉ ở cột 0", run(sol.setZeroes, [[1, 2], [0, 4]]), [[0, 2], [0, 0]])
check("0 chỉ ở hàng 0", run(sol.setZeroes, [[1, 0], [3, 4]]), [[0, 0], [3, 0]])
check("Một hàng", run(sol.setZeroes, [[1, 0, 3]]), [[0, 0, 0]])

random.seed(73)
for _ in range(3000):
    m, n = random.randint(1, 6), random.randint(1, 6)
    mat = [[random.choice([0, 1, 2, 3, 4, 5, -1]) for _ in range(n)] for _ in range(m)]
    expected = run(set_zeroes_copy, mat)
    assert run(sol.setZeroes, mat) == expected == run(set_zeroes_sets, mat), mat
print("3000 ma trận ngẫu nhiên: bản O(1) khớp bản sao và bản dùng set")

Ví dụ 1                PASS  [[1, 0, 1], [0, 0, 0], [1, 0, 1]]
Ví dụ 2                PASS  [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]
1 x 1 là 0             PASS  [[0]]
Không có 0             PASS  [[1, 2], [3, 4]]
0 chỉ ở cột 0          PASS  [[0, 2], [0, 0]]
0 chỉ ở hàng 0         PASS  [[0, 0], [3, 0]]
Một hàng               PASS  [[0, 0, 0]]


3000 ma trận ngẫu nhiên: bản O(1) khớp bản sao và bản dùng set


# 9. Vì sao đúng? Độ phức tạp?

- Sau pha đánh dấu: với `r ≥ 1`, `matrix[r][0] == 0` ⇔ hàng `r` có số 0 gốc (kể cả khi chính `matrix[r][0]` là 0 gốc). Với `c ≥ 1`, `matrix[0][c] == 0` ⇔ cột `c` có số 0 gốc. `matrix[0][0] == 0` ⇔ hàng 0 có số 0 gốc. `first_col_zero` ⇔ cột 0 có số 0 gốc.
- Pha xóa bên trong chỉ **đọc** sổ, chỉ **ghi** vào ô `r, c ≥ 1` → sổ không bị hỏng.
- Hàng 0 và cột 0 được xử lý **sau cùng**, khi sổ đã dùng xong.

Vì sao pha đánh dấu không làm hỏng chính nó? Nó chỉ ghi **số 0** vào hàng 0/cột 0 ở những hàng/cột **thật sự** sẽ bị xóa — nên dù ô đó sau này được đọc như "số 0 gốc", kết luận vẫn đúng.

```text
Time:  O(m · n)
Space: O(1) (bản chính) | O(m + n) (bản set) | O(m · n) (bản sao)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Gán 0 ngay khi duyệt

Như mục 3 — số 0 lan khắp ma trận.

## Lỗi 2: Dùng `matrix[0][0]` cho cả hàng 0 và cột 0

In [16]:
def wrong_shared_corner(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    for r in range(m):
        for c in range(n):
            if matrix[r][c] == 0:
                matrix[r][0] = 0                 # SAI: cột 0 không có biến riêng
                matrix[0][c] = 0
    for r in range(1, m):
        for c in range(1, n):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    if matrix[0][0] == 0:
        for c in range(n):
            matrix[0][c] = 0
        for r in range(m):
            matrix[r][0] = 0

A = [[1, 0, 3], [4, 5, 6], [7, 8, 9]]
B = copy.deepcopy(A)
wrong_shared_corner(A)
print("Sai:"); show(A)
sol.setZeroes(B)
print("Đúng:"); show(B)

Sai:
   0  0  0
   0  0  6
   0  0  9
Đúng:
   0  0  0
   4  0  6
   7  0  9


Số 0 ở `(0, 1)` đánh dấu `matrix[0][0] = 0` (hàng 0 cần xóa — đúng), nhưng bản sai hiểu luôn là cột 0 cần xóa — sai.

## Lỗi 3: Xử lý hàng 0 / cột 0 **trước** phần bên trong

In [17]:
def wrong_order(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    first_col_zero = any(matrix[r][0] == 0 for r in range(m))
    for r in range(m):
        for c in range(1, n):
            if matrix[r][c] == 0:
                matrix[r][0] = 0
                matrix[0][c] = 0
    if matrix[0][0] == 0:                        # SAI: xóa hàng 0 trước -> sổ cột bị toàn 0
        for c in range(n):
            matrix[0][c] = 0
    if first_col_zero:
        for r in range(m):
            matrix[r][0] = 0
    for r in range(1, m):
        for c in range(1, n):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0

A = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
wrong_order(A)
print("Sai:"); show(A)
print("Đúng:"); show(run(sol.setZeroes, [[0, 1, 2], [3, 4, 5], [6, 7, 8]]))

Sai:
   0  0  0
   0  0  0
   0  0  0
Đúng:
   0  0  0
   0  4  5
   0  7  8


## Lỗi 4: Dùng giá trị "đặc biệt" (như `-1` hoặc `None`) để đánh dấu

Phần tử có thể là **bất kỳ** số nguyên 32-bit, nên `-1` có thể là dữ liệu thật. Đánh dấu bằng `None` thì đúng nhưng vẫn là "mẹo" dễ sai kiểu; cách dùng hàng 0 / cột 0 sạch hơn.

# 11. Mẹo nhận diện pattern

**"Dùng chính input làm bộ nhớ phụ"** khi follow-up đòi `O(1)` bộ nhớ:

- Ma trận: hàng 0 / cột 0 làm sổ (bài này), hoặc **mã hóa hai trạng thái** trong một ô (#289 Game of Life: dùng bit thứ 2).
- Mảng: đánh dấu bằng **dấu âm** (`nums[x-1] = -abs(...)`) — #448, #442, #41.

Luôn hỏi: thông tin đánh dấu có **ghi đè** dữ liệu còn cần không? Nếu có, lưu phần bị ghi đè vào vài biến riêng (ở đây là `first_col_zero`).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Áp dụng thuật toán `O(1)` cho ma trận dưới và ghi lại trạng thái sau pha đánh dấu:

```text
1 2 0
4 0 6
7 8 9
```

## Bài 2 — Điền chỗ trống (bản `O(m + n)`)

In [18]:
def exercise_2_set_zeroes(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    zero_rows = [False] * m
    zero_cols = [False] * n
    # TODO: pha 1 — đánh dấu
    # TODO: pha 2 — ghi 0
    pass


# A = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]; exercise_2_set_zeroes(A); assert A == [[1, 0, 1], [0, 0, 0], [1, 0, 1]]
# A = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]; exercise_2_set_zeroes(A); assert A == [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]

## Bài 3 — Biến thể: dùng hàng 0 làm sổ, nhưng biến riêng cho **hàng** 0

Viết lại bản `O(1)` theo cách đối xứng: `matrix[0][0]` dành cho **cột** 0, và một biến `first_row_zero` cho hàng 0.

In [19]:
def exercise_3_set_zeroes_row_flag(matrix: List[List[int]]) -> None:
    # TODO
    pass


# A = [[1, 0, 3], [4, 5, 6], [7, 8, 9]]; exercise_3_set_zeroes_row_flag(A); assert A == [[0, 0, 0], [4, 0, 6], [7, 0, 9]]
# A = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]; exercise_3_set_zeroes_row_flag(A); assert A == [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]

## Bài 4 — Biến thể: đếm ô bị xóa

Không sửa ma trận, trả về **số ô sẽ thành 0** sau `setZeroes` (kể cả các ô vốn đã là 0). `O(m + n)` bộ nhớ, `O(m·n)` thời gian.

```python
count_zeroed([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) -> 5
count_zeroed([[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]) -> 8
```

Gợi ý nhanh: nếu có `R` hàng và `C` cột cần xóa, số ô = `R·n + C·m − R·C` (bao hàm – loại trừ).

In [20]:
def exercise_4_count_zeroed(matrix: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_count_zeroed([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == 5
# assert exercise_4_count_zeroed([[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]) == 8
# assert exercise_4_count_zeroed([[1, 2], [3, 4]]) == 0

## Bài 5 — Mini challenge: Game of Life (LeetCode #289) — cập nhật tại chỗ

Mỗi ô `0` (chết) hoặc `1` (sống). Thế hệ sau (theo 8 hàng xóm): ô sống có `< 2` hoặc `> 3` hàng xóm sống thì chết; có 2 hoặc 3 thì sống tiếp; ô chết có **đúng 3** hàng xóm sống thì sống lại. Cập nhật **tại chỗ**, `O(1)` bộ nhớ phụ.

Gợi ý nhanh: mã hóa hai trạng thái trong một số: bit 0 = trạng thái cũ, bit 1 = trạng thái mới. Đếm hàng xóm bằng `cell & 1`; cuối cùng dịch `cell >>= 1`.

In [21]:
def exercise_5_game_of_life(board: List[List[int]]) -> None:
    # TODO
    pass


# B = [[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]]; exercise_5_game_of_life(B); assert B == [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]
# B = [[1, 1], [1, 0]]; exercise_5_game_of_life(B); assert B == [[1, 1], [1, 1]]

# 13. Gợi ý

**Bài 1**
- Mức 1: `(0,2)=0` → `matrix[0][0]=0`, `matrix[0][2]=0`. `(1,1)=0` → `matrix[1][0]=0`, `matrix[0][1]=0`.
- Mức 2: `first_col_zero = False` (cột 0 ban đầu `1, 4, 7` không có 0).

**Bài 3**
- Mức 1: `first_row_zero = any(x == 0 for x in matrix[0])`.
- Mức 2: quét `r` từ 1, `c` từ 0: gặp 0 thì `matrix[r][0] = 0` và `matrix[0][c] = 0`.
- Mức 3: xóa bên trong, rồi cột 0 (nếu `matrix[0][0] == 0`), rồi hàng 0 (nếu `first_row_zero`).

**Bài 4**
- Mức 1: tính `zero_rows`, `zero_cols` như Bài 2 rồi dùng công thức.

**Bài 5**
- Mức 1: chỉ ghi bit 1 (`cell |= 2`) khi ô **sống ở thế hệ sau**.
- Mức 2: ô sống: sống tiếp nếu `live in (2, 3)`; ô chết: sống lại nếu `live == 3`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [22]:
trace_set_zeroes([[1, 2, 0], [4, 0, 6], [7, 8, 9]])

Ban đầu:
   1  2  0
   4  0  6
   7  8  9
Sau pha đánh dấu (first_col_zero = False):
   0  0  0
   0  0  6
   7  8  9
Sau khi xóa phần bên trong:
   0  0  0
   0  0  0
   7  0  0
Sau khi xử lý hàng 0 và cột 0:
   0  0  0
   0  0  0
   7  0  0


## Đáp án Bài 2

In [23]:
def answer_2_set_zeroes(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    zero_rows = [False] * m
    zero_cols = [False] * n
    for r in range(m):
        for c in range(n):
            if matrix[r][c] == 0:
                zero_rows[r] = zero_cols[c] = True
    for r in range(m):
        for c in range(n):
            if zero_rows[r] or zero_cols[c]:
                matrix[r][c] = 0

A = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]; answer_2_set_zeroes(A); assert A == [[1, 0, 1], [0, 0, 0], [1, 0, 1]]
A = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]; answer_2_set_zeroes(A); assert A == [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [24]:
def answer_3_set_zeroes_row_flag(matrix: List[List[int]]) -> None:
    m, n = len(matrix), len(matrix[0])
    first_row_zero = any(x == 0 for x in matrix[0])
    for r in range(1, m):
        for c in range(n):
            if matrix[r][c] == 0:
                matrix[r][0] = 0
                matrix[0][c] = 0
    for r in range(1, m):
        for c in range(1, n):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    if matrix[0][0] == 0:
        for r in range(m):
            matrix[r][0] = 0
    if first_row_zero:
        for c in range(n):
            matrix[0][c] = 0

A = [[1, 0, 3], [4, 5, 6], [7, 8, 9]]; answer_3_set_zeroes_row_flag(A); assert A == [[0, 0, 0], [4, 0, 6], [7, 0, 9]]
A = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]; answer_3_set_zeroes_row_flag(A); assert A == [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]
for _ in range(2000):
    m, n = random.randint(1, 6), random.randint(1, 6)
    mat = [[random.choice([0, 1, 2, 3]) for _ in range(n)] for _ in range(m)]
    assert run(answer_3_set_zeroes_row_flag, mat) == run(set_zeroes_copy, mat), mat
print("Bài 3 PASS (2000 ma trận ngẫu nhiên)")

Bài 3 PASS (2000 ma trận ngẫu nhiên)

Chú ý: `matrix[0][0]` giờ có nghĩa "cột 0 cần xóa". Ta phải đánh dấu nó **chỉ** từ các số 0 ở hàng `r ≥ 1` của cột 0 — hoặc từ chính `matrix[0][0]` gốc (nếu nó là 0, cả hàng 0 lẫn cột 0 đều xóa, nên vẫn đúng).

## Đáp án Bài 4

In [25]:
def answer_4_count_zeroed(matrix: List[List[int]]) -> int:
    m, n = len(matrix), len(matrix[0])
    rows = {r for r in range(m) for c in range(n) if matrix[r][c] == 0}
    cols = {c for r in range(m) for c in range(n) if matrix[r][c] == 0}
    R, C = len(rows), len(cols)
    return R * n + C * m - R * C

assert answer_4_count_zeroed([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == 5
assert answer_4_count_zeroed([[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]) == 8
assert answer_4_count_zeroed([[1, 2], [3, 4]]) == 0
for _ in range(1000):
    m, n = random.randint(1, 6), random.randint(1, 6)
    mat = [[random.choice([0, 1, 2]) for _ in range(n)] for _ in range(m)]
    assert answer_4_count_zeroed(mat) == sum(x == 0 for row in run(sol.setZeroes, mat) for x in row)
print("Bài 4 PASS")

Bài 4 PASS

## Đáp án Bài 5

In [26]:
def answer_5_game_of_life(board: List[List[int]]) -> None:
    m, n = len(board), len(board[0])
    for r in range(m):
        for c in range(n):
            live = 0
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if (dr or dc) and 0 <= r + dr < m and 0 <= c + dc < n:
                        live += board[r + dr][c + dc] & 1
            if board[r][c] & 1:
                if live in (2, 3):
                    board[r][c] |= 2
            elif live == 3:
                board[r][c] |= 2
    for r in range(m):
        for c in range(n):
            board[r][c] >>= 1

B = [[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]]; answer_5_game_of_life(B); assert B == [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]
B = [[1, 1], [1, 0]]; answer_5_game_of_life(B); assert B == [[1, 1], [1, 1]]
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Hai pha: **đánh dấu** rồi **ghi**. Để đạt `O(1)`, dùng hàng 0 / cột 0 làm sổ, thêm **một biến** cho phần bị dùng chung ở `matrix[0][0]`, và xử lý hàng 0 / cột 0 **sau cùng**.

---

## Checklist trước khi nộp

- [ ] Không gán 0 ngay trong lúc quét?
- [ ] Có biến riêng cho cột 0 (hoặc hàng 0)?
- [ ] Xử lý phần bên trong **trước**, hàng 0 / cột 0 **sau**?
- [ ] Hàm không `return` gì (sửa tại chỗ)?